# 03 · 模型选择与调参

回答两个问题：**用哪一类模型**，以及 **CatBoost 用什么参数和训练方式**。所有比较都使用 01 的固定五折，只看同一折上的配对差。

1. 基线：逻辑回归、XGBoost、CatBoost（CPU / GPU）、AutoGluon
2. 线性模型与随机森林 + 数值变换
3. CatBoost 调参（内层早停、学习率、深度、L2、抽样）
4. CatBoost 训练方式（有序提升、生长方式、抽样、分箱、类别组合）

LightGBM 与神经网络要和最终的 CatBoost 比较融合效果，放在 05。

| notebook | 内容 |
|---|---|
| **01_data_exploration** | 原始数据读取、审计、探索分析、数据质量检查、固定五折 |
| **02_feature_engineering** | 全部特征的定义、三轮特征消融、保存特征库 |
| **03_model_selection** | 基线模型（逻辑回归、XGBoost、CatBoost、AutoGluon）、线性模型的数值变换、CatBoost 调参与训练方式筛选 |
| **04_final_model** | 最终五折模型、多种子、学习曲线、全量重训与最终提交 |
| **05_error_analysis** | 残差诊断、LightGBM 与神经网络、跨实验融合、伪标签、公开榜噪声与运行间差异 |

原始数据在 `data/`；每本 notebook 的结果保存在 `artifacts/` 下与它同名的目录。

In [1]:
# 环境、路径与通用函数（5 本 notebook 相同）
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def find_project_dir():
    candidates = []
    if os.environ.get('HMDA_PROJECT_DIR'):
        candidates.append(Path(os.environ['HMDA_PROJECT_DIR']))
    candidates.append(Path('/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis'))
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'data' / 'train.csv').is_file() and (candidate / 'data' / 'test.csv').is_file():
            return candidate.expanduser().resolve()
    raise FileNotFoundError('找不到项目目录（应包含 data/train.csv 与 data/test.csv）；请设置环境变量 HMDA_PROJECT_DIR')


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / 'data'                                        # 原始 CSV（只读）
ARTIFACTS = PROJECT_DIR / 'artifacts'                                  # 每本 notebook 的结果写入同名子目录
PREPARED_DIR = ARTIFACTS / '01_data_exploration'                       # 01：固定数据接口
FEATURE_DIR = ARTIFACTS / '02_feature_engineering' / 'feature_store'   # 02：特征库
SEED = 42
CATBOOST_VERSION = '1.2.10'
GPU_NAME = None
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})


def ensure_catboost():
    """只在训练时调用：安装并导入固定版本的 CatBoost。"""
    global catboost
    try:
        installed = importlib.metadata.version('catboost')
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != CATBOOST_VERSION:
        if 'catboost' in sys.modules:
            raise RuntimeError('当前运行时已导入另一版 CatBoost；请重启运行时后从头运行')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'catboost=={CATBOOST_VERSION}'])
        importlib.invalidate_caches()
    import catboost as catboost_module
    if catboost_module.__version__ != CATBOOST_VERSION:
        raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}，实际为 {catboost_module.__version__}')
    catboost = catboost_module
    return catboost_module


def ensure_gpu():
    """只在训练时调用：确认 Colab 分配了 NVIDIA GPU。"""
    global GPU_NAME
    if shutil.which('nvidia-smi') is None:
        raise RuntimeError('训练需要 GPU：Colab「更改运行时类型」选 T4 GPU 后从头运行')
    probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           text=True, capture_output=True, check=False)
    if probe.returncode != 0 or not probe.stdout.strip():
        raise RuntimeError('NVIDIA GPU 不可用：请选择 T4 GPU 后从头运行')
    GPU_NAME = probe.stdout.strip().splitlines()[0]
    return GPU_NAME


print(f'项目：{PROJECT_DIR}')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')

Mounted at /content/drive
项目：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis
Python 3.13.16 | pandas 2.2.3 | numpy 2.1.3


**训练开关**：默认全部为 `False`，只读取 `artifacts/` 中已保存的结果并生成表格和图（Colab 普通 CPU 运行时即可，几分钟跑完）。把某一项改为 `True` 并选择 **T4 GPU** 运行时，会按原协议重新训练对应部分；每个模型训练完立即写入 `artifacts/`，断线后从头运行会跳过已完成的部分。

In [2]:
TRAIN = {
    'baseline_cpu': False,   # CatBoost R1/R2 CPU 基线（Colab CPU 上约 3 小时）
    'tuning': False,         # CatBoost 调参五折复核（GPU，约 2.5 小时）
    'screen': False,         # 训练方式筛选（GPU，约 2 小时）
}

## 0. 读取固定数据接口与特征库

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_data_exploration.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = DATA_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    DATA_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    DATA_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

In [ ]:
# 读取 02 生成的特征库（只在重新训练时真正需要；只读结果时缺少也不影响）
def load_feature_store():
    meta_path = FEATURE_DIR / 'feature_store.json'
    require(meta_path.is_file(), '缺少特征库：请先运行 02_feature_engineering.ipynb')
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    require(meta['data_hashes'] == DATA_HASHES, '特征库与 01 的固定数据接口不一致：请重跑 02')
    stores = {}
    for part, expected_rows in [('train', len(X_base)), ('test', len(X_test_base))]:
        path = FEATURE_DIR / f'blocks_{part}.parquet'
        require(sha256_file(path) == meta['files'][path.name], f'{path.name} 哈希不符：请重跑 02')
        frame = pd.read_parquet(path)
        require(len(frame) == expected_rows, f'{path.name} 行数不符')
        for column in frame.columns:
            frame[column] = frame[column].astype(object if column in meta['categorical'] else 'float64')
        stores[part] = frame
    blocks = {name: (stores['train'][cols].reset_index(drop=True), stores['test'][cols].reset_index(drop=True))
              for name, cols in meta['blocks'].items()}
    return blocks, set(meta['categorical']), meta


def variant_frames(blocks):
    """原始 56 列 + 指定的特征块；与 02 中的定义相同。"""
    train_parts = [X_base] + [BLOCKS[b][0] for b in blocks]
    test_parts = [X_test_base] + [BLOCKS[b][1] for b in blocks]
    train = pd.concat(train_parts, axis=1)
    test = pd.concat(test_parts, axis=1)
    require(train.columns.is_unique and train.columns.equals(test.columns), '变体列不一致')
    require(train.iloc[:, :len(FEATURE_COLUMNS)].equals(X_base), '原始 56 列被修改')
    cats = CATEGORICAL_COLUMNS + [c for c in train.columns if c in NEW_CATEGORICAL]
    return train, test, cats


if (FEATURE_DIR / 'feature_store.json').is_file():
    BLOCKS, NEW_CATEGORICAL, FEATURE_META = load_feature_store()
    print(f"特征库：{len(BLOCKS)} 个特征块、{sum(len(c) for c in FEATURE_META['blocks'].values())} 列；"
          f"特征集 {list(FEATURE_META['feature_sets'])}")
else:
    BLOCKS, NEW_CATEGORICAL, FEATURE_META = None, set(), None
    print('尚未生成特征库（02）：只读取已保存结果不受影响；需要重新训练时请先运行 02。')

In [ ]:
# 只读的结果工具：读取每个（模型/变体, 折）保存的 JSON，做同折配对比较
def load_runs(directory, pattern='*__fold?.json'):
    directory = Path(directory)
    paths = sorted(directory.glob(pattern))
    require(paths, f'{directory} 中没有结果文件：请确认 artifacts 已从 Drive 同步，或打开对应的训练开关')
    return pd.DataFrame([json.loads(p.read_text(encoding='utf-8')) for p in paths])


def paired_table(runs, key, baseline, min_gain=0.0005, min_folds=4):
    auc = runs.pivot(index='fold', columns=key, values='valid_auc')
    require(baseline in auc.columns, f'缺少基线 {baseline}')
    delta = auc.sub(auc[baseline], axis=0)
    table = pd.DataFrame({
        'folds': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'delta_vs_base': delta.mean(),
        'folds_better': (delta > 0).sum(),
    })
    table['keep'] = ((table['folds'] == 5) & (table['delta_vs_base'] >= min_gain)
                     & (table['folds_better'] >= min_folds))
    table.loc[baseline, 'keep'] = False
    return table.sort_values('mean_auc', ascending=False), delta


def plot_deltas(delta, baseline, title):
    others = delta.drop(columns=[baseline])
    order = others.mean().sort_values().index.tolist()
    means = others[order].mean() * 1e4
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(order) + 1.6))
    ax.barh(range(len(order)), means, color=['#4c72b0' if v > 0 else '#c44e52' for v in means])
    for i, name in enumerate(order):
        ax.scatter(others[name] * 1e4, np.full(len(others), i), color='black', s=12, zorder=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(range(len(order)), order)
    ax.set_xlabel(f'AUC change vs {baseline} (x1e-4); bar = 5-fold mean, dots = single folds')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## 1. 基线模型

| 模型 | 设置 |
|---|---|
| 逻辑回归 | 地理简化（N2）+ DTI×贷款类型交互 + LTV 尾部标记，折内填补与标准化 |
| XGBoost | 原生类别，深度 6，折内早停 |
| CatBoost R1 / R2 | 原始 56 列，深度 6 / 8，学习率 0.05，L2=3，固定 1600 轮 |

CatBoost 明显最好。**同参数下 GPU 版比 CPU 版低约 0.001**——第 3 节会看到原因主要是两者默认的行抽样方式不同。逻辑回归与 XGBoost 的完整训练代码见 Git 历史第一个提交中的 `02_hmda_models.ipynb`；CatBoost 基线可用下方开关重训。

In [ ]:
cpu_run = ARTIFACTS / '03_model_selection' / 'baseline_catboost_xgboost'   # CatBoost R1/R2、XGBoost（CPU）
lr_run = ARTIFACTS / '03_model_selection' / 'baseline_logistic'            # 逻辑回归
rows = []
for path in [cpu_run / 'summary.csv', lr_run / 'summary.csv', cpu_run / 'blend_summary.csv']:
    for r in pd.read_csv(path).itertuples():
        rows.append({'模型': r.model, '五折 AUC': r.mean_fold_auc, '标准差': r.std_fold_auc,
                     'OOF LogLoss': r.pooled_oof_logloss})
rows.append({'模型': 'CatBoost_R2（GPU T4，同参数）', '五折 AUC': 0.91498, '标准差': np.nan, 'OOF LogLoss': np.nan})
display(pd.DataFrame(rows).sort_values('五折 AUC', ascending=False).reset_index(drop=True).round(5))

In [ ]:
if TRAIN['baseline_cpu']:
    ensure_catboost()
    # 重新训练 CatBoost R1 / R2（CPU，固定 1600 轮；与原基线协议相同），结果写入单独目录，不覆盖原结果
    RERUN_DIR = ARTIFACTS / '03_model_selection' / 'baseline_catboost_rerun'
    RERUN_DIR.mkdir(parents=True, exist_ok=True)
    base_params = dict(iterations=1600, learning_rate=0.05, l2_leaf_reg=3.0, loss_function='Logloss',
                       eval_metric='AUC', nan_mode='Min', random_seed=SEED, task_type='CPU',
                       thread_count=os.cpu_count() or 2, use_best_model=False,
                       allow_writing_files=False, verbose=False)
    rerun_rows = []
    for fold in range(1, 6):
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        for name, depth in [('CatBoost_R1', 6), ('CatBoost_R2', 8)]:
            path = RERUN_DIR / f'{name}__fold{fold}.json'
            if path.is_file():
                rerun_rows.append(json.loads(path.read_text(encoding='utf-8')))
                continue
            started = time.perf_counter()
            model = catboost.CatBoostClassifier(**base_params, depth=depth)
            model.fit(X_base.iloc[fit_idx], y.iloc[fit_idx], cat_features=CATEGORICAL_COLUMNS)
            probability = checked_probabilities(model, X_base.iloc[valid_idx])
            row = {'model': name, 'fold': fold, 'valid_auc': roc_auc_score(y.iloc[valid_idx], probability),
                   'seconds': time.perf_counter() - started}
            write_json(path, row)
            rerun_rows.append(row)
            print(f"{name} fold {fold}: AUC={row['valid_auc']:.6f}  {row['seconds'] / 60:.1f} 分钟", flush=True)
    display(pd.DataFrame(rerun_rows).pivot(index='fold', columns='model', values='valid_auc').agg(['mean', 'std']).round(6))
else:
    print("训练开关关闭（TRAIN['baseline_cpu'] = False）：本节只读取 artifacts/ 中已保存的结果。")

**AutoGluon**（`extreme_quality`、T4 GPU、2,400 秒，只跑第 1 折）：0.8996。TabICL、TabDPT 因内存不足失败，实际只训练出一个 LightGBM；这不代表 AutoGluon 的上限，但在可用资源下没有竞争力。

## 2. 线性模型、随机森林与数值变换

在 Ridge 与随机森林上比较 6 种数值表示（原值、二阶、三阶、分位数正态、RBF 近似、样条），类别统一 One-Hot。变换对 Ridge 最多 +0.003、对随机森林无益，两者都远低于 CatBoost（≈ 0.916），因此后续只用 CatBoost。原始训练代码见 Git 历史第一个提交中的 `07_hmda_transformations_comparison.ipynb`。

In [ ]:
TRANSFORM_DIR = ARTIFACTS / '03_model_selection' / 'transformations'
transform = pd.read_csv(TRANSFORM_DIR / 'auc_summary.csv')
order = ['base', 'poly2', 'poly3', 'quantile', 'rbf', 'spline']
display(transform.pivot(index='representation', columns='model', values='auc_mean').reindex(order).round(5))
if (TRANSFORM_DIR / 'auc_comparison.png').is_file():
    display(Image(filename=str(TRANSFORM_DIR / 'auc_comparison.png'), width=760))

## 3. CatBoost 调参

**协议**：外层训练部分内再留出 15% 早停，得到最佳轮数 *b*，再在整个外层训练部分按 `round(b / 0.85)` 轮重训；外层验证只用于评分。B0 = GPU 上的 R2（学习率 0.05、固定 1600 轮）。先在第 1 折上筛 12 组，再对前几名做五折复核。

In [ ]:
EXPERIMENT_ID = 'catboost_tuning'
FOLDS = [1, 2, 3, 4, 5]     # 第二轮：五折复核；只做第 1 折筛选时改回 [1]
RUN_ONLY = ['B0_R2_fixed1600', 'C7_d8_lr03_l2_10_sub80',
            'C10_d8_lr03_l2_30_sub80', 'C11_d6_lr03_l2_10_sub80']  # None 表示全部候选
INNER_STOP_FRACTION = 0.15
MAX_ITERATIONS = 10000
OD_WAIT = 300
OUTPUT_DIR = ARTIFACTS / '03_model_selection' / EXPERIMENT_ID

GPU_BASE = dict(
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    allow_writing_files=False, verbose=False,
)
# early_stop=False 表示固定轮数、不早停；按预计耗时从小到大排列
CANDIDATES = {
    'B0_R2_fixed1600':   dict(depth=8, learning_rate=0.05, l2_leaf_reg=3, early_stop=False, iterations=1600),
    'C1_d8_lr05_l2_3_es': dict(depth=8, learning_rate=0.05, l2_leaf_reg=3),
    'C6_d6_lr03_l2_10':  dict(depth=6, learning_rate=0.03, l2_leaf_reg=10),
    'C3_d8_lr03_l2_10':  dict(depth=8, learning_rate=0.03, l2_leaf_reg=10),
    'C2_d8_lr03_l2_3':   dict(depth=8, learning_rate=0.03, l2_leaf_reg=3),
    'C4_d8_lr03_l2_30':  dict(depth=8, learning_rate=0.03, l2_leaf_reg=30),
    'C7_d8_lr03_l2_10_sub80': dict(depth=8, learning_rate=0.03, l2_leaf_reg=10,
                                   bootstrap_type='Bernoulli', subsample=0.8),
    'C8_d8_lr03_l2_10_DTI': dict(depth=8, learning_rate=0.03, l2_leaf_reg=10, dti=True),
    'C5_d10_lr03_l2_10': dict(depth=10, learning_rate=0.03, l2_leaf_reg=10),
    'C9_d8_lr02_l2_10':  dict(depth=8, learning_rate=0.02, l2_leaf_reg=10),
    # 第二轮：把第 1 折前三名（抽样、depth 6、L2=30）组合起来
    'C10_d8_lr03_l2_30_sub80': dict(depth=8, learning_rate=0.03, l2_leaf_reg=30,
                                    bootstrap_type='Bernoulli', subsample=0.8),
    'C11_d6_lr03_l2_10_sub80': dict(depth=6, learning_rate=0.03, l2_leaf_reg=10,
                                    bootstrap_type='Bernoulli', subsample=0.8),
}
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(CANDIDATES)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知候选：{sorted(unknown)}')

In [ ]:
if TRAIN['tuning']:
    ensure_catboost()
    ensure_gpu()
    X_dti = pd.concat([X_base, BLOCKS['dti'][0]], axis=1)   # C8 使用的 DTI 上下界
    fixed_config(OUTPUT_DIR, {
        'purpose': 'GPU CatBoost screening, inner 15% early stop then outer refit',
        'seed': SEED, 'inner_stop_fraction': INNER_STOP_FRACTION,
        'max_iterations': MAX_ITERATIONS, 'od_wait': OD_WAIT,
        'gpu_base': {k: v for k, v in GPU_BASE.items() if k != 'thread_count'},
        'data_hashes': DATA_HASHES,
    })


    def candidate_params(spec, iterations):
        params = {k: v for k, v in spec.items() if k not in {'early_stop', 'dti', 'iterations'}}
        return dict(GPU_BASE, **params, iterations=iterations)


    def run_one(name, spec, fold):
        features = X_dti if spec.get('dti') else X_base
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        started = time.perf_counter()
        best_iteration = None
        early_stopped = None
        if spec.get('early_stop', True):
            inner_fit, inner_stop = train_test_split(
                fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED,
                stratify=y.iloc[fit_idx],
            )
            probe = catboost.CatBoostClassifier(**candidate_params(spec, MAX_ITERATIONS),
                                                eval_metric='Logloss', od_type='Iter',
                                                od_wait=OD_WAIT, use_best_model=True)
            probe.fit(features.iloc[inner_fit], y.iloc[inner_fit],
                      cat_features=CATEGORICAL_COLUMNS,
                      eval_set=(features.iloc[inner_stop], y.iloc[inner_stop]))
            best_iteration = int(probe.get_best_iteration()) + 1
            early_stopped = int(probe.tree_count_) < MAX_ITERATIONS
            if not early_stopped:
                print(f'  注意：{name} 到 {MAX_ITERATIONS} 轮仍未早停，结果可能偏保守')
            rounds = int(round(best_iteration / (1 - INNER_STOP_FRACTION)))
            del probe
            gc.collect()
        else:
            rounds = int(spec['iterations'])
        model = catboost.CatBoostClassifier(**candidate_params(spec, rounds), use_best_model=False)
        model.fit(features.iloc[fit_idx], y.iloc[fit_idx], cat_features=CATEGORICAL_COLUMNS)
        require(int(model.tree_count_) == rounds, '外层重训轮数不符')
        probability = checked_probabilities(model, features.iloc[valid_idx])
        actual = y.iloc[valid_idx]
        pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                      'prob': probability}).to_csv(
            OUTPUT_DIR / f'{name}__fold{fold}.csv', index=False, float_format='%.12f')
        result = {
            'candidate': name, 'fold': fold,
            'spec': {k: v for k, v in spec.items()},
            'feature_count': features.shape[1],
            'inner_best_iteration': best_iteration, 'early_stopped': early_stopped, 'refit_rounds': rounds,
            'valid_auc': roc_auc_score(actual, probability),
            'valid_logloss': log_loss(actual, probability, labels=[0, 1]),
            'valid_brier': brier_score_loss(actual, probability),
            'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
            'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        write_json(OUTPUT_DIR / f'{name}__fold{fold}.json', result)
        del model
        gc.collect()
        return result


    def load_results():
        rows = [json.loads(p.read_text(encoding='utf-8'))
                for p in sorted(OUTPUT_DIR.glob('*__fold*.json'))]
        return pd.DataFrame(rows)


    selected = {n: s for n, s in CANDIDATES.items() if RUN_ONLY is None or n in RUN_ONLY}
    for fold in FOLDS:
        for name, spec in selected.items():
            result_path = OUTPUT_DIR / f'{name}__fold{fold}.json'
            if result_path.is_file():
                saved = json.loads(result_path.read_text(encoding='utf-8'))
                require(saved['spec'] == spec, f'{name} 的定义已改变；请改候选名')
                print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                continue
            result = run_one(name, spec, fold)
            print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  '
                  f'轮数={result["refit_rounds"]}  {result["seconds"]/60:.1f} 分钟', flush=True)

    results = load_results()
    require(not results.empty, '还没有完成的结果')
    auc = results.pivot(index='fold', columns='candidate', values='valid_auc')
    require('B0_R2_fixed1600' in auc.columns, '缺少对照 B0_R2_fixed1600')
    delta = auc.sub(auc['B0_R2_fixed1600'], axis=0)
    table = pd.DataFrame({
        'folds_done': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'mean_delta_vs_B0': delta.mean(),
        'folds_better_than_B0': (delta > 0).sum(),
        'mean_refit_rounds': results.groupby('candidate')['refit_rounds'].mean(),
        'mean_logloss': results.groupby('candidate')['valid_logloss'].mean(),
        'total_minutes': results.groupby('candidate')['seconds'].sum() / 60,
    }).sort_values('mean_delta_vs_B0', ascending=False)
    table.to_csv(OUTPUT_DIR / 'summary.csv')
    display(table.round(6))
    display(delta.round(6))
else:
    print("训练开关关闭（TRAIN['tuning'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
TUNING_DIR = ARTIFACTS / '03_model_selection' / 'catboost_tuning'
tuning_runs = load_runs(TUNING_DIR)
fold1 = tuning_runs[tuning_runs['fold'].eq(1)].set_index('candidate')
screen = pd.DataFrame({
    '第 1 折 AUC': fold1['valid_auc'],
    '相对 B0': fold1['valid_auc'] - fold1.loc['B0_R2_fixed1600', 'valid_auc'],
    '重训轮数': fold1['refit_rounds'],
}).sort_values('第 1 折 AUC', ascending=False)
print('第一轮：第 1 折筛选')
display(screen.round(6))
finalists = ['B0_R2_fixed1600', 'C7_d8_lr03_l2_10_sub80', 'C10_d8_lr03_l2_30_sub80', 'C11_d6_lr03_l2_10_sub80']
table, delta = paired_table(tuning_runs[tuning_runs['candidate'].isin(finalists)], 'candidate', 'B0_R2_fixed1600')
print('第二轮：前几名五折复核')
display(table.round(6))
plot_deltas(delta, 'B0_R2_fixed1600', 'CatBoost tuning, 5 folds (vs R2 on GPU)')

**结论**：**C7（depth 8、lr 0.03、L2=10、Bernoulli 抽样 0.8）+0.00156** 与 **C11（depth 6、其余相同）+0.00142**，均 5/5 折。只加 Bernoulli 抽样一项就贡献约 +0.001——GPU 默认 Bayesian 抽样、CPU 默认 MVS 抽样，这正是 GPU 版原来低约 0.001 的原因。depth 10 更差。最终模型使用 C7 与 C11。

## 4. CatBoost 训练方式筛选

在最终特征（LEAN，68 列）上，检验 GPU 默认设置之外的选项：有序提升（为小数据设计，超过 5 万行时 GPU 默认关闭）、NewtonL2 打分、Depthwise / Lossguide 生长、Poisson 抽样、抽样 0.66、254 个分箱、不做类别组合、删除重复的人口统计列。基线 V0 ≈ 04 的 M3，学习率 0.06 加速；每折内层早停后直接给外层验证打分。保留标准：平均 ≥ +0.0005 且 ≥ 4/5 折。

In [ ]:
EXPERIMENT_ID = 'training_modes'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None                 # 例如 ['V0_base', 'V1_ordered']；None 表示全部
INNER_STOP_FRACTION = 0.15
OUTPUT_DIR = ARTIFACTS / '03_model_selection' / EXPERIMENT_ID

NO_FREQ = ['dti', 'ltv', 'income', 'missing']
LEAN = NO_FREQ + ['precision', 'thresholds']        # M3 的 68 列特征
DEMO_DUPLICATES = [                                 # 与 derived_* 重复的人口统计原始编码与「观察」标记
    'applicant_ethnicity-1', 'co-applicant_ethnicity-1',
    'applicant_race-1', 'co-applicant_race-1',
    'applicant_ethnicity_observed', 'co-applicant_ethnicity_observed',
    'applicant_race_observed', 'co-applicant_race_observed',
    'applicant_sex_observed', 'co-applicant_sex_observed',
]
SCREEN_BASE = dict(              # 接近 M3（C11）但学习率加倍以加快筛选；外层验证只评分
    depth=6, learning_rate=0.06, l2_leaf_reg=10,
    bootstrap_type='Bernoulli', subsample=0.8,
    loss_function='Logloss', eval_metric='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    iterations=6000, od_type='Iter', od_wait=200, use_best_model=True,
    allow_writing_files=False, verbose=False,
)
VARIANTS = {
    'V0_base': {},
    'V1_ordered': {'boosting_type': 'Ordered'},            # 超过 5 万行时 GPU 默认 Plain，从未试过
    'V2_newtonL2': {'score_function': 'NewtonL2'},         # 默认 Cosine
    'V3_depthwise': {'grow_policy': 'Depthwise', 'depth': 7, 'min_data_in_leaf': 50},
    'V4_lossguide': {'grow_policy': 'Lossguide', 'depth': 10, 'max_leaves': 48,
                     'min_data_in_leaf': 50, 'score_function': 'NewtonL2'},
    'V5_poisson': {'bootstrap_type': 'Poisson', 'subsample': 0.8},   # 仅 GPU 支持
    'V6_sub066': {'subsample': 0.66},                      # 更强的行抽样
    'V7_border254': {'border_count': 254},                 # GPU 默认 128 个数值分箱
    'V8_ctr1': {'max_ctr_complexity': 1},                  # 不做类别组合，降低方差
    'V9_drop_demo': {'drop': DEMO_DUPLICATES},             # 删去 10 个重复的人口统计列
}
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(VARIANTS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知变体：{sorted(unknown)}')

In [ ]:
if TRAIN['screen']:
    ensure_catboost()
    ensure_gpu()
    lean_train, lean_test, lean_cats = variant_frames(LEAN)


    def frames_for(spec):
        drop = spec.get('drop', [])
        unknown = set(drop) - set(lean_train.columns)
        require(not unknown, f'要删除的列不存在：{sorted(unknown)}')
        columns = [c for c in lean_train.columns if c not in drop]
        return lean_train[columns], [c for c in lean_cats if c not in drop]


    def params_for(spec, **extra):
        params = dict(SCREEN_BASE)
        params.update({k: v for k, v in spec.items() if k != 'drop'})
        params.update(extra)
        return params


    # 预检：每个变体在 3,000 行上训练 20 轮，几秒内暴露 GPU 不支持的参数组合，避免跑了一小时才报错
    rng = np.random.default_rng(SEED)
    probe_rows = rng.choice(len(y), 3000, replace=False)
    failed = {}
    for name, spec in VARIANTS.items():
        train, cats = frames_for(spec)
        try:
            model = catboost.CatBoostClassifier(**params_for(spec, iterations=20))
            model.fit(train.iloc[probe_rows[:2500]], y.iloc[probe_rows[:2500]], cat_features=cats,
                      eval_set=(train.iloc[probe_rows[2500:]], y.iloc[probe_rows[2500:]]))
            checked_probabilities(model, train.iloc[probe_rows[2500:]])
            print(f'预检通过：{name}（{train.shape[1]} 列）')
        except Exception as exc:
            failed[name] = repr(exc)
            print(f'预检失败，跳过：{name} → {exc!r}')
    require('V0_base' not in failed, '基线 V0_base 预检失败，请检查环境')

    fixed_config(OUTPUT_DIR, {
        'purpose': 'data-efficiency screen on M3 features; inner 15% early stop, score probe model on outer fold',
        'seed': SEED, 'inner_stop_fraction': INNER_STOP_FRACTION,
        'screen_base': {k: v for k, v in SCREEN_BASE.items() if k != 'thread_count'},
        'features': LEAN, 'data_hashes': DATA_HASHES,
    })
    selected = {n: s for n, s in VARIANTS.items()
                if (RUN_ONLY is None or n in RUN_ONLY) and n not in failed}
    for fold in FOLDS:
        fit_idx = np.flatnonzero(fold_assignment != fold)
        valid_idx = np.flatnonzero(fold_assignment == fold)
        inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION,
                                                 random_state=SEED, stratify=y.iloc[fit_idx])
        for name, spec in selected.items():
            path = OUTPUT_DIR / f'{name}__fold{fold}.json'
            if path.is_file():
                saved = json.loads(path.read_text(encoding='utf-8'))
                require(saved['spec'] == spec, f'{name} 的定义已改变；请改变体名')
                print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
                continue
            train, cats = frames_for(spec)
            started = time.perf_counter()
            model = catboost.CatBoostClassifier(**params_for(spec))
            model.fit(train.iloc[inner_fit], y.iloc[inner_fit], cat_features=cats,
                      eval_set=(train.iloc[inner_stop], y.iloc[inner_stop]))
            best = int(model.get_best_iteration()) + 1
            probability = checked_probabilities(model, train.iloc[valid_idx])
            actual = y.iloc[valid_idx]
            pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                          'prob': probability}).to_csv(OUTPUT_DIR / f'{name}__fold{fold}__valid.csv',
                                                       index=False, float_format='%.12f')
            result = {'variant': name, 'fold': fold, 'spec': spec, 'feature_count': train.shape[1],
                      'best_iteration': best,
                      'hit_iteration_limit': best >= SCREEN_BASE['iterations'] - SCREEN_BASE['od_wait'],
                      'valid_auc': roc_auc_score(actual, probability),
                      'valid_logloss': log_loss(actual, probability, labels=[0, 1]),
                      'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
                      'finished_at_utc': datetime.now(timezone.utc).isoformat()}
            write_json(path, result)
            print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  最佳轮数={best}  '
                  f'{result["seconds"]/60:.1f} 分钟', flush=True)
            del model
            gc.collect()

    results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8'))
                            for p in sorted(OUTPUT_DIR.glob('*__fold?.json'))])
    require(not results.empty, '还没有完成的结果')
    auc = results.pivot(index='fold', columns='variant', values='valid_auc')
    require('V0_base' in auc.columns, '缺少基线 V0_base')
    delta = auc.sub(auc['V0_base'], axis=0)
    table = pd.DataFrame({
        'folds_done': auc.notna().sum(),
        'mean_auc': auc.mean(),
        'mean_delta_vs_V0': delta.mean(),
        'folds_better_than_V0': (delta > 0).sum(),
        'mean_best_iteration': results.groupby('variant')['best_iteration'].mean(),
        'total_minutes': results.groupby('variant')['seconds'].sum() / 60,
    })
    table['keep'] = ((table['folds_done'] == 5) & (table['mean_delta_vs_V0'] >= 0.0005)
                     & (table['folds_better_than_V0'] >= 4))
    table.loc['V0_base', 'keep'] = False
    table = table.sort_values('mean_delta_vs_V0', ascending=False)
    table.to_csv(OUTPUT_DIR / 'summary.csv')
    display(table.round(6))
    display(delta.round(6))
    if failed:
        print('预检失败、未运行的变体：', failed)
    kept = table.index[table['keep']].tolist()
    print('达到保留标准（≥ +0.0005 且 ≥ 4/5 折）的变体：', kept if kept else '无')
else:
    print("训练开关关闭（TRAIN['screen'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
screen_runs = load_runs(ARTIFACTS / '03_model_selection' / 'training_modes')
table, delta = paired_table(screen_runs, 'variant', 'V0_base')
display(table.round(6))
plot_deltas(delta, 'V0_base', 'CatBoost training modes, 5 folds')

**结论**：没有一项达标；254 个分箱 +0.00023（4/5）最好，其余持平或更差。现有设置（对称树、Plain 提升、Cosine 打分、Bernoulli 0.8）已是这些选项中最好的。

## 5. 小结

最终模型为 **CatBoost（GPU）**，参数 **C7 / C11**；其他训练方式都没有带来可靠提升。04 把这两组参数与 02 的特征集组合成最终模型；LightGBM 与神经网络能否与它互补，在 05 中检验。